# 🧪 FINAL TEST — весь проект в одном ноутбуке (запусти один раз)

Этот ноутбук — **рабочая мини-версия всего проекта**, чтобы ты понял как он устроен.
Нажми «Run All» — и он сам пройдёт весь путь:

1. 🏗️ **Сгенерирует игрушечные данные** (здания «до/после» для разных катастроф) — *качать ничего не надо!*
2. 👁️ Покажет примеры картинок
3. 🧱 Обучит **CNN с нуля** (siamese)
4. 🏋️ Обучит **предобученную** модель (ResNet18, transfer learning)
5. 🎯 Протестирует обе на **невиданной катастрофе** (held-out) и сравнит
6. 🗺️ Нарисует **карту повреждений**

> **📥 Нужно ли качать данные?** НЕТ. Данные генерируются внутри ноутбука.
> Единственное, что может скачаться из интернета — веса ResNet18 (~45 МБ, автоматически, один раз).
> Нет интернета? Не страшно — код сам откатится на случайную инициализацию и всё равно отработает.

> **⏱️ Сколько ждать?** На CPU примерно **3–6 минут** суммарно (CNN с нуля < 1 мин, ResNet ~2–4 мин).
> На GPU — около 30 секунд. Точное время ноутбук печатает сам.

> ⚠️ Это **синтетика для демонстрации механики**. Цифры точности тут игрушечные.
> Реальный проект использует датасет xBD и общий код в `src/`. Здесь всё в одном файле специально,
> чтобы читалось сверху вниз как одна история.

## 0 · Установка библиотек
Если чего-то не хватает — ячейка доустановит. `torch` большой (~200 МБ), первый раз может качаться пару минут.

In [ ]:
import importlib.util, subprocess, sys
for pkg in ['numpy', 'matplotlib', 'torch', 'torchvision']:
    if importlib.util.find_spec(pkg) is None:
        print(f'Installing {pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg])
print('Libraries ready ✅')

## 1 · Импорты и настройки
Всё конфигурируется здесь: размер картинки, классы повреждений, какие катастрофы для обучения
и какую прячем для теста. `wildfire` (пожар) — наша held-out катастрофа: модель её НИКОГДА не видит при обучении.

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

PATCH = 64                                   # размер картинки здания (px)
CLASSES = ['no-damage', 'minor', 'major', 'destroyed']   # 4 уровня урона
TRAIN_DISASTERS = ['hurricane', 'flood', 'earthquake']   # учимся на этих
HELDOUT = 'wildfire'                          # тестируем на этой (невиданной!)

print('Устройство для обучения:', device)
print('Учимся на:', TRAIN_DISASTERS, '| Прячем для теста:', HELDOUT)

## 2 · Генерируем игрушечные данные 🏗️
Каждый «патч» = маленькая картинка одного здания. Мы делаем **пару**: `pre` (до) и `post` (после).

- **Уровень урона** управляет тем, *как много* пикселей здания изменилось (0% → 90%).
- **Тип катастрофы** управляет тем, *как выглядит* повреждение (цветом):
  наводнение = синее, пожар = оранжево-чёрное, ураган/землетрясение = серое.

Это и есть суть проекта в миниатюре: модель учится ловить **изменение** между до/после.
А поскольку у пожара (held-out) цвет повреждений другой — мы увидим, что точность на нём проседает.
Это и есть тот самый **transfer gap**, ради которого весь проект.

In [ ]:
DISASTER_COLOR = {
    'hurricane':  [0.50, 0.50, 0.55],   # серо-голубой мусор
    'flood':      [0.20, 0.30, 0.80],   # вода = синий
    'earthquake': [0.45, 0.42, 0.40],   # серые обломки
    'wildfire':   [0.75, 0.35, 0.08],   # гарь = оранжево-чёрный
}
DAMAGE_FRAC = {0: 0.0, 1: 0.20, 2: 0.50, 3: 0.90}   # доля изменённых пикселей здания

def make_pair(damage, disaster, rng):
    """Вернуть (pre, post) — две картинки HxWx3 в диапазоне [0,1]."""
    # фон (земля) + шум
    base = np.ones((PATCH, PATCH, 3), np.float32) * np.array([0.55, 0.60, 0.45], np.float32)
    base += rng.normal(0, 0.04, base.shape).astype(np.float32)
    # прямоугольник-здание в случайном месте
    h, w = rng.integers(PATCH//3, PATCH//2, size=2)
    y, x = rng.integers(0, PATCH - h), rng.integers(0, PATCH - w)
    roof = np.array([0.60, 0.60, 0.62], np.float32) + rng.normal(0, 0.03, 3).astype(np.float32)
    pre = base.copy(); pre[y:y+h, x:x+w] = roof
    # post = pre, но часть здания заменена 'сигнатурой' катастрофы
    post = pre.copy()
    mask = rng.random((PATCH, PATCH)) < DAMAGE_FRAC[damage]
    region = np.zeros((PATCH, PATCH), bool); region[y:y+h, x:x+w] = True
    mask &= region
    sig = np.array(DISASTER_COLOR[disaster], np.float32)
    post[mask] = sig + rng.normal(0, 0.05, (mask.sum(), 3)).astype(np.float32)
    return np.clip(pre, 0, 1), np.clip(post, 0, 1)

def build_split(n, disasters, rng):
    """Сгенерировать n патчей. Классы несбалансированы (как в реальности: большинство целые)."""
    pre_list, post_list, labels = [], [], []
    class_p = [0.55, 0.20, 0.15, 0.10]   # дисбаланс: много 'no-damage'
    for _ in range(n):
        d = rng.choice(4, p=class_p)
        dis = rng.choice(disasters)
        pre, post = make_pair(d, dis, rng)
        pre_list.append(pre); post_list.append(post); labels.append(d)
    X_pre = np.transpose(np.array(pre_list), (0, 3, 1, 2))    # -> N,C,H,W
    X_post = np.transpose(np.array(post_list), (0, 3, 1, 2))
    return X_pre.astype(np.float32), X_post.astype(np.float32), np.array(labels)

rng = np.random.default_rng(SEED)
# train/val — только обучающие катастрофы; test — ТОЛЬКО held-out (pozhar)
Xtr_pre, Xtr_post, ytr = build_split(900, TRAIN_DISASTERS, rng)
Xva_pre, Xva_post, yva = build_split(200, TRAIN_DISASTERS, rng)
Xte_pre, Xte_post, yte = build_split(300, [HELDOUT], rng)
print('train:', len(ytr), '| val:', len(yva), '| test (held-out wildfire):', len(yte))
print('Баланс классов в train:', np.bincount(ytr))

## 3 · Посмотрим на данные глазами 👁️
Верхний ряд — `pre` (до), нижний — `post` (после). Подпись = уровень урона.
Видно: чем сильнее урон, тем больше «грязи» на здании.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
rng_show = np.random.default_rng(0)
for col, d in enumerate([0, 1, 2, 3]):
    idx = np.where(ytr == d)[0][0]
    axes[0, col].imshow(np.transpose(Xtr_pre[idx], (1, 2, 0)))
    axes[0, col].set_title(f'PRE  ({CLASSES[d]})')
    axes[1, col].imshow(np.transpose(Xtr_post[idx], (1, 2, 0)))
    axes[1, col].set_title(f'POST ({CLASSES[d]})')
for ax in axes.ravel(): ax.axis('off')
plt.tight_layout(); plt.show()

## 4 · Готовим данные для PyTorch
Нормализуем картинки (как ждёт ResNet) и заворачиваем в `DataLoader`, который выдаёт данные пачками (батчами).

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(1, 3, 1, 1)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(1, 3, 1, 1)
def norm(x): return (x - IMAGENET_MEAN) / IMAGENET_STD

def make_loader(Xpre, Xpost, y, shuffle):
    ds = TensorDataset(torch.tensor(norm(Xpre)), torch.tensor(norm(Xpost)), torch.tensor(y))
    return DataLoader(ds, batch_size=64, shuffle=shuffle)

train_dl = make_loader(Xtr_pre, Xtr_post, ytr, True)
val_dl   = make_loader(Xva_pre, Xva_post, yva, False)
test_dl  = make_loader(Xte_pre, Xte_post, yte, False)
print('Батчей в train:', len(train_dl))

## 5 · Две модели 🤖
Обе **siamese (две ветки с общими весами)**: одна и та же сеть смотрит на `pre` и на `post`,
получаются два вектора-описания. Мы их объединяем как `[pre, post, |pre−post|]` —
абсолютная разница `|pre−post|` это буквально сигнал *изменения*. Дальше маленькая «голова» выдаёт 4 класса.

- **ScratchCNN** — маленькая сеть с нуля.
- **PretrainedCNN** — та же идея, но ветка = ResNet18, предобученный на миллионах картинок.

In [ ]:
def combine(f1, f2):
    return torch.cat([f1, f2, torch.abs(f1 - f2)], dim=1)

class ScratchCNN(nn.Module):
    def __init__(self, n_classes=4, feat=128):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                                 nn.MaxPool2d(2))
        self.enc = nn.Sequential(block(3, 32), block(32, 64), block(64, 128),
                                 nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                 nn.Linear(128, feat), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(feat*3, 128), nn.ReLU(), nn.Dropout(0.3),
                                  nn.Linear(128, n_classes))
    def forward(self, pre, post):
        return self.head(combine(self.enc(pre), self.enc(post)))

from torchvision.models import resnet18, ResNet18_Weights
class PretrainedCNN(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        try:
            net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)   # качает ~45МБ один раз
            print('ResNet18: загружены предобученные веса ✅')
        except Exception as e:
            net = resnet18(weights=None)
            print('ResNet18: нет интернета -> случайные веса (демо всё равно отработает) ⚠️')
        feat = net.fc.in_features; net.fc = nn.Identity()
        # заморозим раннее, дообучим позднее (быстрее + меньше переобучения)
        for i, child in enumerate(net.children()):
            if i < 6:
                for p in child.parameters(): p.requires_grad = False
        self.enc = net
        self.head = nn.Sequential(nn.Linear(feat*3, 256), nn.ReLU(), nn.Dropout(0.4),
                                  nn.Linear(256, n_classes))
    def forward(self, pre, post):
        return self.head(combine(self.enc(pre), self.enc(post)))
print('Модели определены.')

## 6 · Обучение 🏋️
Один общий цикл обучения для обеих моделей — так сравнение честное.
Используем **веса классов** в функции потерь, чтобы модель не ленилась и не ставила всем 'no-damage'
(помнишь дисбаланс?). Время каждой тренировки печатается.

In [ ]:
# веса классов: редкий класс -> больший вес
counts = np.bincount(ytr, minlength=4)
class_weights = torch.tensor(counts.sum() / (4 * counts), dtype=torch.float32).to(device)
print('Веса классов:', class_weights.cpu().numpy().round(2))

def train(model, epochs=5, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    t0 = time.time()
    for ep in range(epochs):
        model.train(); tot = 0
        for pre, post, y in train_dl:
            pre, post, y = pre.to(device), post.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(pre, post), y); loss.backward(); opt.step()
            tot += loss.item() * len(y)
        # быстрая валидация
        model.eval(); correct = n = 0
        with torch.no_grad():
            for pre, post, y in val_dl:
                out = model(pre.to(device), post.to(device))
                correct += (out.argmax(1).cpu() == y).sum().item(); n += len(y)
        print(f'  epoch {ep+1}/{epochs}  loss {tot/len(ytr):.3f}  val_acc {correct/n:.2f}')
    print(f'  ⏱️ обучение заняло {time.time()-t0:.1f} сек')
    return model

In [ ]:
print('=== CNN с нуля ===')
scratch = train(ScratchCNN())

In [ ]:
print('=== Предобученная (ResNet18) ===')
pretrained = train(PretrainedCNN())

## 7 · Оценка: seen vs held-out 🎯  (главный результат)
Сравниваем обе модели на двух наборах:
- **val** — те же катастрофы, что в обучении (модель их «видела»)
- **test** — пожар, которого модель НИКОГДА не видела (held-out)

Смотрим не только на общую точность, но и на **матрицу ошибок** (confusion matrix) и macro-F1.
Скорее всего точность на held-out будет ниже — это и есть transfer gap.

In [ ]:
@torch.no_grad()
def predict(model, dl):
    model.eval(); ys, ps = [], []
    for pre, post, y in dl:
        out = model(pre.to(device), post.to(device))
        ys += y.tolist(); ps += out.argmax(1).cpu().tolist()
    return np.array(ys), np.array(ps)

def report(name, y, p):
    cm = np.zeros((4, 4), int)
    for t, q in zip(y, p): cm[t, q] += 1
    acc = np.trace(cm) / cm.sum()
    f1s = []
    for c in range(4):
        tp = cm[c, c]; prec = tp / max(cm[:, c].sum(), 1); rec = tp / max(cm[c, :].sum(), 1)
        f1s.append(2*prec*rec / max(prec+rec, 1e-9))
    print(f'{name:32s} acc={acc:.2f}  macro-F1={np.mean(f1s):.2f}')
    return cm, acc

results = {}
for label, model in [('scratch', scratch), ('pretrained', pretrained)]:
    for split, dl in [('SEEN (val)', val_dl), ('HELD-OUT (wildfire)', test_dl)]:
        y, p = predict(model, dl)
        cm, acc = report(f'{label} | {split}', y, p)
        results[(label, split)] = cm

In [ ]:
# матрицы ошибок: строки = правда, столбцы = предсказание. Идеал = всё на диагонали.
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, ((label, split), cm) in zip(axes.ravel(), results.items()):
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(4)); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
    ax.set_yticks(range(4)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel('предсказано'); ax.set_ylabel('правда')
    ax.set_title(f'{label}\n{split}')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

## 8 · Карта повреждений 🗺️
Финальный шаг проекта: берём «снимок района» (сетку зданий), предсказываем урон для каждого
и раскрашиваем — 🟩 зелёный = цел, 🟥 красный = разрушен. Так выглядел бы наш результат для спасателей.

In [ ]:
GRID = 6   # район 6x6 зданий
rng_map = np.random.default_rng(7)
pre_tiles, post_tiles, true_d = [], [], []
for _ in range(GRID*GRID):
    d = rng_map.choice(4, p=[0.55, 0.20, 0.15, 0.10])
    pre, post = make_pair(d, HELDOUT, rng_map)   # район пострадал от пожара (held-out)
    pre_tiles.append(pre); post_tiles.append(post); true_d.append(d)
Xp = np.transpose(np.array(post_tiles), (0, 3, 1, 2)).astype(np.float32)
Xq = np.transpose(np.array(pre_tiles),  (0, 3, 1, 2)).astype(np.float32)
with torch.no_grad():
    out = pretrained(torch.tensor(norm(Xq)).to(device), torch.tensor(norm(Xp)).to(device))
    pred_d = out.argmax(1).cpu().numpy()

DAMAGE_RGB = np.array([[0.1,0.7,0.1],[0.9,0.9,0.1],[0.95,0.5,0.1],[0.85,0.1,0.1]])  # зел->красн
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
# мозаика 'после'
mosaic = np.ones((GRID*PATCH, GRID*PATCH, 3), np.float32)
for k, tile in enumerate(post_tiles):
    r, c = divmod(k, GRID); mosaic[r*PATCH:(r+1)*PATCH, c*PATCH:(c+1)*PATCH] = tile
axes[0].imshow(mosaic); axes[0].set_title('Снимок ПОСЛЕ (район)'); axes[0].axis('off')
# карта истинного урона и предсказанного
for ax, dmg, title in [(axes[1], true_d, 'Истинный урон'), (axes[2], pred_d, 'Предсказание модели')]:
    grid_img = DAMAGE_RGB[np.array(dmg).reshape(GRID, GRID)]
    ax.imshow(grid_img); ax.set_title(title); ax.axis('off')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print('Точность карты:', (pred_d == np.array(true_d)).mean().round(2))
print('Легенда: зелёный=цел, жёлтый=лёгкий, оранжевый=сильный, красный=разрушен')

## ✅ Готово!
Ты только что прогнал **весь проект в миниатюре**:
данные → две модели → обучение → оценка на невиданной катастрофе → карта повреждений.

**Что здесь было ненастоящим:** данные (рисованные) и потому игрушечные цифры.
**Что настоящее:** вся механика — siamese-архитектура, transfer learning, held-out оценка, веса классов, карта.

**Дальше по-настоящему:**
1. Role A качает xBD и делает `manifest.csv` (ноутбук `01`)
2. Тот же код моделей (он уже лежит в `src/models.py`) учится на реальных патчах
3. Held-out катастрофа = настоящие пожары вместо нарисованных

Вопросы по любому шагу — спрашивай! 🚀